In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Rdd").getOrCreate()
sc=spark.sparkContext
print("spark is ready !!")

spark is ready !!


In [ ]:
num=[1,2,3,4,5]

num_rdd=sc.parallelize(num)
print(num_rdd)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297


In [ ]:
num_rdd.collect()

[1, 2, 3, 4, 5]

In [ ]:
Names=["Niki","Praki","Tharun"]
name_rdd=sc.parallelize(Names)
print(name_rdd)

ParallelCollectionRDD[1] at readRDDFromFile at PythonRDD.scala:297


In [ ]:
name_rdd.collect()

['Niki', 'Praki', 'Tharun']

In [7]:
name_rdd.getNumPartitions()

2

In [8]:
name_rdd.glom().collect()

[['Niki'], ['Praki', 'Tharun']]

In [9]:
rdd=sc.parallelize([10,20,30],4)
rdd.glom().collect()

[[], [10], [20], [30]]

**Lazy evaluation:**
transformation ,action
transformation only gets executed when it sees action .it backtrack it and executes  needed block of code .

**RDD operations**: two categories
1. **Transformations**
These create a new RDD.

Examples:

map
filter
flatMap
distinct
union
intersection
groupByKey
reduceByKey
2.** Actions**
These return a result to the driver.

Examples:

collect
count
first
take
reduce

In [10]:
rdd_num=sc.parallelize([1,2,3,4,5])
mapped_rdd=rdd.map(lambda x:x*2)
filtered_rdd=mapped_rdd.filter(lambda x:x>5)
print("nothing gets executed")

nothing gets executed


In [12]:
filtered_rdd.collect()

[20, 40, 60]

collect-action sees it and backtrack and executes transformation block

In [13]:
squared_rdd=rdd.map(lambda x:x*x)
squared_rdd.collect()

[100, 400, 900]

In [16]:
rdd=sc.parallelize(["rahul","sneha","arjun"])
upper_rdd=rdd.map(lambda x:x.upper())
upper_rdd.collect()

['RAHUL', 'SNEHA', 'ARJUN']

**map** allows to iterate

In [18]:
rdd=sc.parallelize([10,15,20,25,30])
even_rdd=rdd.filter(lambda x:x%2==0)
even_rdd.collect()

[10, 20, 30]

In [19]:
rdd=sc.parallelize(["apple","banana","kiwi"])
filtered_rdd=rdd.filter(lambda x:len(x)>4)
filtered_rdd.collect()

['apple', 'banana']

filter -returns filtered record to driver.
Map can catch only oen element -flatmap-can catch multiple element

In [20]:
lines=sc.parallelize([
    "Niki is beautiful",
    "Praki is intelligent"
])
filtered_lines=lines.flatMap(lambda x:x.split(" "))
filtered_lines.collect()


['Niki', 'is', 'beautiful', 'Praki', 'is', 'intelligent']

map-takes each line searately like word count
flatmap-takes it as only one

In [21]:
result=lines.map(lambda x:x.split(" "))
result.collect()

[['Niki', 'is', 'beautiful'], ['Praki', 'is', 'intelligent']]

In [23]:
result=lines.flatMap(lambda x:x.split(" "))
result.collect()

['Niki', 'is', 'beautiful', 'Praki', 'is', 'intelligent']

In [24]:
rdd=sc.parallelize([10,20,30,30,40,40])
rdd.distinct().collect()

[10, 20, 30, 40]

In [25]:
rdd1=sc.parallelize([1,2,3])
rdd2=sc.parallelize([3,4,5])
rdd1.union(rdd2).collect()

[1, 2, 3, 3, 4, 5]

In [26]:
rdd1.intersection(rdd2).collect()

[3]

In [29]:
rdd1.subtract(rdd2).collect()

[1, 2]

**Action:**

In [30]:
rdd=sc.parallelize([10,20,30])
rdd.collect()

[10, 20, 30]

In [31]:
rdd.count()

3

In [32]:
rdd.first()

10

In [33]:
rdd.take(3)

[10, 20, 30]

In [35]:
rdd.reduce(lambda a,b:a+b)


60

key values:
group by key
reduce by key

In [36]:
data=[
    ("IT",70000),
    ("HR",60000),
    ("IT",75000),
    ("Finance",80000),
    ("HR",58000)
]
rdd=sc.parallelize(data)
rdd.collect()

[('IT', 70000),
 ('HR', 60000),
 ('IT', 75000),
 ('Finance', 80000),
 ('HR', 58000)]

In [37]:
rdd.reduceByKey(lambda a,b:a+b).collect()

[('IT', 145000), ('HR', 118000), ('Finance', 80000)]

In [38]:
grouped=rdd.groupByKey()
[(k,list(v)) for k,v in grouped.collect()]

[('IT', [70000, 75000]), ('HR', [60000, 58000]), ('Finance', [80000])]

In [39]:
rdd.mapValues(lambda x:x+5).collect()


[('IT', 70005),
 ('HR', 60005),
 ('IT', 75005),
 ('Finance', 80005),
 ('HR', 58005)]

based on key : sortBy,sortByKey

In [40]:
rdd.sortBy(lambda x:x).collect()

[('Finance', 80000),
 ('HR', 58000),
 ('HR', 60000),
 ('IT', 70000),
 ('IT', 75000)]

based on values sort:

In [44]:
rdd.sortBy(lambda x:x[1]).collect()

[('HR', 58000),
 ('HR', 60000),
 ('IT', 70000),
 ('IT', 75000),
 ('Finance', 80000)]

Word count in multiple sentence : in map line it just add each etter coma 1 ex (python,1),(spark,1),(python,1).....

In [45]:
lines=sc.parallelize([
    "python spark python",
    "spark hadoop spark",
    "python hadoop"
])
word_count=(
    lines
    .flatMap(lambda x:x.split(" "))
    .map(lambda x:(x,1))
    .reduceByKey(lambda a,b:a+b)
)
word_count.collect()

[('python', 3), ('hadoop', 2), ('spark', 3)]

high salary collect:

In [47]:
employees=sc.parallelize([
    ("Rahul",70000),
    ("Sneha",60000),
    ("Arjun",75000),
    ("Priya",80000),
    ("Karan",50000)
])

high_salary=employees.filter(lambda x:x[1]>65000)
high_salary.collect()

[('Rahul', 70000), ('Arjun', 75000), ('Priya', 80000)]